In [1]:
folders = [
    "homer_simpson", "ned_flanders", "moe_szyslak", "lisa_simpson", "bart_simpson",
    "marge_simpson", "krusty_the_clown", "charles_montgomery_burns", "principal_skinner",
    "milhouse_van_houten", "chief_wiggum", "abraham_grampa_simpson", "sideshow_bob",
    "apu_nahasapeemapetilon", "kent_brockman", "edna_krabappel", "comic_book_guy",
    "nelson_muntz", "lenny_leonard", "mayor_quimby"
]
name2num = {ch: i for i, ch in enumerate(folders)}
num2name = {i: ch for ch, i in name2num.items()}

In [2]:
from torchvision import transforms as t
from PIL import Image

transform = t.Compose([
    t.Resize((224, 224)),
    t.ToTensor(),
    t.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def prep(path_img: str):
    img = Image.open(path_img).convert('RGB')
    return transform(img)

In [3]:
import numpy as np
import os
import torch


class My_Dataset:
    def __init__(self, dataset_path: str = r'/home/d.golomolzin/AI_1lab/simpsons/train_dataset'):
        self.X, self.y = [], []

        for folder in folders:
            folder_path = os.path.join(dataset_path, folder)
            
            imgs = os.listdir(folder_path)
            
            for img_name in imgs:
                self.X.append(os.path.join(folder_path, img_name))
                self.y.append(name2num[folder])

    def __len__(self):
        return len(self.X)

    def __getitem__(self, key):
        return prep(self.X[key]), self.y[key]

In [4]:
from torch.utils.data import DataLoader

batch = 32

dataloader_train = DataLoader(My_Dataset(), batch_size=batch, num_workers=4, shuffle=True, pin_memory=True)

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights
from torch.optim import Adam
import torch.nn as nn


model = resnet18(weights=ResNet18_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(512, 20)
model.to('cuda')

In [6]:
loss_func = nn.CrossEntropyLoss()
optimizer = Adam(model.fc.parameters())

total = len(dataloader_train.dataset)

model.train()
for epoch in range(15):
    curr_loss = 0.0
    for xb, yb in dataloader_train:

        xb, yb = xb.to('cuda'), yb.to('cuda')

        optimizer.zero_grad()
        pred = model(xb)

        loss = loss_func(pred, yb)
        loss.backward()
        optimizer.step()

        curr_loss += loss.item() * xb.size(dim=0)
        
    print(f'Epoch {epoch+1}, loss: {curr_loss/total:.5f}')

Epoch 1, loss: 2.37367
Epoch 2, loss: 1.52993
Epoch 3, loss: 1.21745
Epoch 4, loss: 1.05027
Epoch 5, loss: 0.93673
Epoch 6, loss: 0.86301
Epoch 7, loss: 0.81375
Epoch 8, loss: 0.76096
Epoch 9, loss: 0.71126
Epoch 10, loss: 0.69511
Epoch 11, loss: 0.65688
Epoch 12, loss: 0.63601
Epoch 13, loss: 0.60526
Epoch 14, loss: 0.59039
Epoch 15, loss: 0.58216


In [9]:
for param in model.parameters():
    param.requires_grad = True

optimizer = Adam(model.parameters(), lr=1e-5)

model.train()
for epoch in range(10):
    curr_loss = 0.0
    for xb, yb in dataloader_train:
        
        xb, yb = xb.to('cuda'), yb.to('cuda')

        optimizer.zero_grad()
        pred = model(xb)

        loss = loss_func(pred, yb)
        loss.backward()

        optimizer.step()

        curr_loss += loss.item() * xb.size(dim=0)

    print(f'Epoch {epoch+1}, loss: {curr_loss/total:.5f}')

Epoch 1, loss: 0.08175
Epoch 2, loss: 0.03327
Epoch 3, loss: 0.02080
Epoch 4, loss: 0.01339
Epoch 5, loss: 0.00917
Epoch 6, loss: 0.00715
Epoch 7, loss: 0.00570
Epoch 8, loss: 0.00447
Epoch 9, loss: 0.00366
Epoch 10, loss: 0.00313


In [10]:
import re

class Test_Dataset:
    def __init__(self, test_path: str = r'/home/d.golomolzin/AI_1lab/simpsons/test_dataset'):
        self.test_path = test_path
        self.imgs = sorted(os.listdir(self.test_path))

        self.X_test = []
        self.y_test = []

        for path_img in self.imgs:
                
            name = re.sub(r'_\d+\.jpg$', '', path_img)
            
            self.X_test.append(os.path.join(self.test_path, path_img))
            self.y_test.append(name2num[name])

    def __len__(self):
        return len(self.X_test)

    def __getitem__(self, key):
        return prep(self.X_test[key]), self.y_test[key]


In [11]:
dataloader_test = DataLoader(Test_Dataset(), batch_size=batch, shuffle=False, num_workers=0, pin_memory=True)

In [14]:
total = len(dataloader_test.dataset)

model.eval()
with torch.no_grad():
    total_correct = 0
    curr_loss = 0.0
    for xb, yb in dataloader_test:

        xb, yb = xb.to('cuda'), yb.to('cuda')

        pred = model(xb)
        
        correct = (torch.argmax(pred, dim=1) == yb).sum().item()
        total_correct += correct

        curr_loss += loss_func(pred, yb).item() * xb.size(dim=0)
        
        print(f'Batch acc: {correct / xb.size(dim=0)}')

print(f'\nFinal All acc: {total_correct / total:.5f}\nFinal loss: {curr_loss/total:.5f}')

Batch acc: 0.90625
Batch acc: 0.96875
Batch acc: 0.96875
Batch acc: 0.90625
Batch acc: 0.9375
Batch acc: 0.9375
Batch acc: 0.90625
Batch acc: 0.96875
Batch acc: 0.96875
Batch acc: 0.875
Batch acc: 0.96875
Batch acc: 0.9375
Batch acc: 0.875
Batch acc: 0.96875
Batch acc: 1.0
Batch acc: 0.9375
Batch acc: 1.0
Batch acc: 0.9375
Batch acc: 0.875
Batch acc: 0.84375
Batch acc: 1.0
Batch acc: 1.0
Batch acc: 0.90625
Batch acc: 0.875
Batch acc: 0.90625
Batch acc: 1.0
Batch acc: 0.9375
Batch acc: 1.0
Batch acc: 0.875
Batch acc: 0.9375
Batch acc: 0.9666666666666667

Final All acc: 0.93838
Final loss: 0.21324
